In [1]:
import mne
import numpy as np
from pathlib import Path 
from scipy.io import loadmat
from mne.preprocessing import EOGRegression
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [2]:
mne.set_log_level('WARNING')

In [3]:
data_folder = Path(r'D:\Coding\BCI project\data')
subjects = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']
ch_renamed = {
    'EEG-Fz':'Fz',
    'EEG-0':'FC3',
    'EEG-1':'FC1',
    'EEG-2':'FCz',
    'EEG-3':'FC2',
    'EEG-4':'FC4',
    'EEG-5':'C5',
    'EEG-C3':'C3',
    'EEG-6':'C1',
    'EEG-Cz':'Cz',
    'EEG-7':'C2',
    'EEG-C4':'C4',
    'EEG-8':'C6',
    'EEG-9':'CP3',
    'EEG-10':'CP1',
    'EEG-11':'CPz',
    'EEG-12':'CP2',
    'EEG-13':'CP4',
    'EEG-14':'P1',
    'EEG-Pz':'Pz',
    'EEG-15':'P2',
    'EEG-16':'POz',
}

In [4]:
def load_test_labels(subject):
    mat_files = loadmat(Path(r'D:\Coding\BCI project\data/true labels') / f'{subject}E.mat')
    all_test_labels = mat_files['classlabel'].flatten()
    mask = (all_test_labels == 1) | (all_test_labels == 2)
    labels_test = all_test_labels[mask] - 1
    return labels_test, mask

In [5]:
def preprocessing(raw, ch_names):
    raw.rename_channels(ch_names)
    raw.set_channel_types({
        'EOG-left':'eog',
        'EOG-central':'eog',
        'EOG-right':'eog'
    })
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing = 'ignore')
    raw.filter(l_freq = 1, h_freq = None, fir_design='firwin')
    raw.set_eeg_reference()
    return raw

In [6]:
def calibration_eog(raw, subject):
    if subject == 'A04':
        calibration = raw.copy().crop(0, 60)
    else:
        calibration = raw.copy().crop(0, 300)
    return calibration

In [7]:
def regression_eog(raw, calibration):
    model_plain = EOGRegression(picks='eeg', picks_artifact='eog').fit(calibration)
    raw_clean_plain = model_plain.apply(raw)
    return raw_clean_plain

In [8]:
def filter_bank(raw):
    raw_dict = {}
    frequency_bands = [(4,8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 36), (36, 40)]
    for l_freq, h_freq in frequency_bands:
        raw_dict[(l_freq, h_freq)] = raw.copy().filter(l_freq, h_freq, fir_design='firwin')
    return raw_dict

In [9]:
def epoching_train(raw_dict):
    epochs_dict = {}
    epochs_cropped_dict = {}
    epochs_data_dict = {}
    epochs_cropped_data_dict = {}
    
    for band in raw_dict:
        
        events, event_id = mne.events_from_annotations(raw_dict[band])
        epochs =  mne.Epochs(raw_dict[band], events, 
                            event_id = {'left hand':event_id['769'], 'right hand':event_id['770']},
                            tmin = -1, tmax = 4, picks = 'eeg', baseline = None, 
                            preload = True)
        epochs_cropped = epochs.copy().crop(tmin = 1.0, tmax = 2)
        epochs_data = epochs.get_data(copy = False)
        epochs_cropped_data = epochs_cropped.get_data(copy = False)
        labels = epochs.events[:, -1] - event_id['769']

        epochs_dict[band] = epochs
        epochs_cropped_dict[band] = epochs_cropped
        epochs_data_dict[band] = epochs_data
        epochs_cropped_data_dict[band] = epochs_cropped_data

    return epochs_dict, epochs_cropped_dict, epochs_data_dict, epochs_cropped_data_dict, labels

In [10]:
def epoching_test(raw_dict):
    epochs_dict = {}
    epochs_cropped_dict = {}
    epochs_data_dict = {}
    epochs_cropped_data_dict = {}

    for band in raw_dict:
        
        events, event_id = mne.events_from_annotations(raw_dict[band])
        epochs = mne.Epochs(raw_dict[band], events, event_id = {'unknown':event_id['783']}, tmin = -1,
                            tmax = 4, picks = 'eeg', baseline = None, preload = True)
        epochs_cropped = epochs.copy().crop(tmin = 1.0, tmax = 2)
        epochs_data = epochs.get_data(copy = False)
        epochs_cropped_data = epochs_cropped.get_data(copy = False)

        epochs_dict[band] = epochs
        epochs_cropped_dict[band] = epochs_cropped
        epochs_data_dict[band] = epochs_data
        epochs_cropped_data_dict[band] = epochs_cropped_data
        
    return epochs_dict, epochs_cropped_dict, epochs_data_dict, epochs_cropped_data_dict

In [11]:
results = {}
class_balance = []

for subject in subjects:
    
    train_files = data_folder / f'{subject}T.gdf' 
    test_files = data_folder / f'{subject}E.gdf'
    raw_train = mne.io.read_raw_gdf(train_files, preload = True)
    raw_test = mne.io.read_raw_gdf(test_files,  preload = True)

    labels_test, mask = load_test_labels(subject)

    raw_train = preprocessing(raw_train, ch_renamed)
    raw_test = preprocessing(raw_test, ch_renamed)

    calibration_train = calibration_eog(raw_train, subject)
    calibration_test = calibration_eog(raw_test, subject)

    raw_train = regression_eog(raw_train, calibration_train)
    raw_test = regression_eog(raw_test, calibration_test)

    raw_train_dict = filter_bank(raw_train)
    raw_test_dict = filter_bank(raw_test)

    epochs_train_dict, epochs_train_cropped_dict, epochs_train_data_dict, epochs_train_cropped_data_dict, labels_train = epoching_train(raw_train_dict)
    epochs_test_dict, epochs_test_cropped_dict, epochs_test_data_dict, epochs_test_cropped_data_dict = epoching_test(raw_test_dict)
    
    epochs_test_cropped_data_masked_dict = {}
    for band, data in epochs_test_cropped_data_dict.items():
        data = data[mask]
        epochs_test_cropped_data_masked_dict[band] = data
        
    X_train_list = []
    X_test_list = []
    csp_dict = {}
    lda = LinearDiscriminantAnalysis()
    selector = SelectKBest(score_func=mutual_info_classif, k = 12)
    
    for band in epochs_train_cropped_data_dict:
        csp = CSP(n_components = 4, reg='ledoit_wolf', log = True, norm_trace = False)
        csp_dict[band] = csp
        X_train = csp.fit_transform(epochs_train_cropped_data_dict[band], labels_train)
        X_train_list.append(X_train)
    X_train_concatenated = np.concatenate(X_train_list, axis = 1)
    X_train_selected = selector.fit_transform(X_train_concatenated, labels_train)
    
    for band in epochs_test_cropped_data_masked_dict:
        X_test = csp_dict[band].transform(epochs_test_cropped_data_masked_dict[band])
        X_test_list.append(X_test)
    X_test_concatenated = np.concatenate(X_test_list, axis = 1)
    X_test_selected = selector.transform(X_test_concatenated)
    
    lda.fit(X_train_selected, labels_train)
    score = lda.score(X_test_selected, labels_test)
    results[subject] = score
    class_balance.append(np.mean(labels_test == labels_test[0]))

final_score = np.mean(list(results.values()))
final_std = np.std(list(results.values()))
class_balance_mean = np.mean(class_balance)
chance_level = max(class_balance_mean, 1 - class_balance_mean)

C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not

In [12]:
print(f'Total classification precision is {final_score * 100}%')
print(f'Standard deviation is {final_std}')
print(f'Chance level is {chance_level * 100}%')
for sbjct, scr in results.items():
    print(f'subject {sbjct} - score {scr}')

Total classification precision is 78.47222222222221%
Standard deviation is 0.1336592595671486
Chance level is 50.0%
subject A01 - score 0.8194444444444444
subject A02 - score 0.5555555555555556
subject A03 - score 0.8958333333333334
subject A04 - score 0.6736111111111112
subject A05 - score 0.875
subject A06 - score 0.5902777777777778
subject A07 - score 0.8194444444444444
subject A08 - score 0.9305555555555556
subject A09 - score 0.9027777777777778
